# 03 · Modeling — HDBSCAN + classifier

Author: Chris Kim — 2026-05, ChainSense · UCSD DS3

Clustering and labeling were run on Google Colab (A100 GPU), not this machine. The code cells below are pasted from that Colab notebook; see the note on why the hardware is required.


## Why this ran on an A100

HDBSCAN doesn't assign points to centroids like k-means — it reconstructs the density structure of the whole dataset: pairwise distances → a mutual-reachability graph → a minimum spanning tree → a condensed cluster hierarchy. The distance/MST stage is roughly O(n²) in time and memory, so 6.3M wallets × 26 features is far past what a laptop holds, and even aggressive subsampling distorts the density estimates HDBSCAN depends on. A Colab A100 (40 GB, GPU-accelerated via RAPIDS cuML) makes the full-population fit feasible.


## Load model-ready features

Read wallet_features_v5_model.parquet — the standardized, address-less matrix from notebook 02.


In [ ]:
# (paste from Colab)


## Fit HDBSCAN (cuML, A100)

Density clustering over all 6.3M wallets; yields a cluster label and membership strength per wallet.


In [ ]:
# (paste from Colab)


## Clusters → archetype families

Inspect each cluster's feature profile and map cluster id → family name (cluster_archetypes.json).


In [ ]:
# (paste from Colab)


## XGBoost classifier for unclustered wallets

Train on the HDBSCAN labels so noise / brand-new wallets get a predicted cluster (xgb_cluster_clf.joblib, scaler_v5.joblib).


In [ ]:
# (paste from Colab)


## Build the wallet lookup table

Join the HDBSCAN and XGBoost results into wallet_lut.parquet (address → effective cluster + label_source) — the table the backend serves.


In [ ]:
# (paste from Colab)
